# Python File Handling: The Complete Guide

**Kajola Gbenga**  
Data Scientist & AI Engineer

---

## What this notebook covers

1. **Reading and writing files** (text files, binary files, file modes, the `with` statement, file pointers, `pathlib`)
2. **Working with CSV and JSON** (`csv` module, `DictReader`/`DictWriter`, `json` module, custom serialization)
3. **Error handling with `try`/`except`** (`else`, `finally`, exception hierarchy, custom exceptions, a full mini project)

## How to use this notebook

- Run the cells **from top to bottom**. Later cells reuse files created by earlier ones.
- Every code cell is followed by a markdown cell with a **Line-by-line breakdown** (the numbered points follow the code from top to bottom and explain what each line does and why) and an **Interpreting the output** section (what the printed result means).
- All files are created inside a folder called `file_handling_lab`, so nothing else on your computer is touched. The last cell deletes that folder.

## Table of contents

| Part | Topic |
|------|-------|
| 0 | Setup |
| 1 | Reading and writing files |
| 2 | Working with CSV |
| 3 | Working with JSON |
| 4 | Error handling with try/except |
| 5 | Mini project: CSV to JSON pipeline |
| 6 | Best practices, cheat sheet and exercises |

# Part 0: Setup

Before touching any file we import the tools we need and create a dedicated sandbox folder. Working inside a sandbox is a good habit: your experiments can never overwrite real data.

In [1]:
import csv
import json
from pathlib import Path

WORKDIR = Path("file_handling_lab")
WORKDIR.mkdir(exist_ok=True)
print("Working folder:", WORKDIR.name)
print("Exists:", WORKDIR.exists())

Working folder: file_handling_lab
Exists: True


### Line-by-line breakdown

1. `import csv` - loads Python's built-in `csv` module, used in Part 2 to read and write comma-separated files correctly (it handles quotes and commas inside values for you).
2. `import json` - loads the built-in `json` module, used in Part 3 to convert between Python objects and JSON text.
3. `from pathlib import Path` - imports the `Path` class, a modern object-oriented way to represent file paths that works the same on Windows, macOS and Linux.
4. *(blank line)* - purely for readability.
5. `WORKDIR = Path("file_handling_lab")` - creates a `Path` object pointing to a folder named `file_handling_lab` in the current directory. Nothing is created on disk yet; this is only a path description. The name is in capitals because it is a constant we will reuse.
6. `WORKDIR.mkdir(exist_ok=True)` - actually creates the folder. `exist_ok=True` means "if it already exists, do not raise an error", so you can re-run the cell safely.
7. `print("Working folder:", WORKDIR.name)` - `.name` returns only the final part of the path (the folder name), and `print` shows it.
8. `print("Exists:", WORKDIR.exists())` - `.exists()` returns `True` or `False` depending on whether the path is present on disk. This confirms step 6 worked.

### Interpreting the output

You should see `Working folder: file_handling_lab` and `Exists: True`. The `True` proves the sandbox folder was created and is ready. If you re-run the cell, the output is identical because of `exist_ok=True`.

---
# Part 1: Reading and Writing Files

## 1.1 The `open()` function and file modes

Every file operation begins with `open(path, mode, encoding=...)`. The **mode** tells Python what you intend to do.

| Mode | Meaning | If file exists | If file does not exist |
|------|---------|----------------|------------------------|
| `"r"` | Read text (default) | Opens it | `FileNotFoundError` |
| `"w"` | Write text | **Erases everything**, then writes | Creates it |
| `"a"` | Append text | Adds to the end | Creates it |
| `"x"` | Exclusive create | `FileExistsError` | Creates it |
| `"r+"` | Read and write | Opens it, no erase | `FileNotFoundError` |
| `"b"` | Add to any mode for **binary** (`"rb"`, `"wb"`, `"ab"`) | same as base mode | same as base mode |

**Always** pass `encoding="utf-8"` for text files. If you skip it, Python uses the operating system default, and the same code can behave differently on different computers.

## 1.2 The `with` statement (context manager)

`with open(...) as f:` guarantees the file is **closed automatically** when the indented block ends, even if an error occurs inside it. Forgetting to close files can lose unwritten data and leak system resources, so `with` is the standard, recommended way.

In [2]:
file_path = WORKDIR / "greetings.txt"

with open(file_path, "w", encoding="utf-8") as f:
    f.write("Hello, Python!\n")
    f.write("File handling is fun.\n")
    chars = f.write("Third line here.\n")

print("Characters written by the last call:", chars)
print("Is the file closed after the with block?", f.closed)

Characters written by the last call: 17
Is the file closed after the with block? True


### Line-by-line breakdown

1. `file_path = WORKDIR / "greetings.txt"` - the `/` operator on a `Path` joins path parts. The result is `file_handling_lab/greetings.txt` with the correct separator for your operating system.
2. *(blank line)*
3. `with open(file_path, "w", encoding="utf-8") as f:` - opens the file in **write mode**. If `greetings.txt` does not exist it is created; if it does, its old content is wiped. The open file object is given the name `f`. The colon starts the block that controls the file's lifetime.
4. `f.write("Hello, Python!\n")` - writes one string to the file. `\n` is the newline character; without it, all text would land on one line, because `write()` never adds a newline by itself.
5. `f.write("File handling is fun.\n")` - writes the second line right after the first.
6. `chars = f.write("Third line here.\n")` - `write()` **returns the number of characters written**. We capture it in `chars` to demonstrate that.
7. *(blank line)* - ends the `with` block. At this moment Python flushes buffered data to disk and closes the file.
8. `print("Characters written by the last call:", chars)` - shows the value returned by the last `write()`.
9. `print("Is the file closed after the with block?", f.closed)` - the variable `f` still exists after the block, but the file it points to is closed. `.closed` is `True` when a file is closed.

### Interpreting the output

`Characters written by the last call: 17` - the text `Third line here.` has 16 characters plus 1 for the newline, so 17. The second line prints `True`, proving the `with` statement closed the file for us without an explicit `f.close()`.

## 1.3 Reading a whole file with `read()`

`read()` returns the **entire file as one string**. It is simple and great for small files, but avoid it for huge files because everything is loaded into memory at once.

In [3]:
with open(file_path, "r", encoding="utf-8") as f:
    content = f.read()

print(content)
print("Type:", type(content))
print("Total characters:", len(content))

Hello, Python!
File handling is fun.
Third line here.

Type: <class 'str'>
Total characters: 54


### Line-by-line breakdown

1. `with open(file_path, "r", encoding="utf-8") as f:` - opens the file we just wrote in **read mode**. `"r"` is the default, so writing it explicitly is optional, but being explicit makes code easier to read.
2. `content = f.read()` - reads everything from the current position (the start) to the end and stores it as a single string.
3. *(blank line)* - block ends, file closes. Note we use `content` **after** the block; the data is already safely in memory.
4. `print(content)` - prints the text. The three lines appear on separate lines because the `\n` characters inside the string are real line breaks.
5. `print("Type:", type(content))` - confirms we received a `str`.
6. `print("Total characters:", len(content))` - `len()` counts every character, including the newline characters.

### Interpreting the output

The three lines of text appear, followed by an empty line (the final `\n` in the file plus the newline `print` adds). `Type: <class 'str'>` shows `read()` gives a plain string. The total is **54**: 15 characters in line one, 22 in line two and 17 in line three (each including its `\n`).

## 1.4 Reading line by line: `readline()`, `readlines()` and iteration

| Method | Returns | Best for |
|--------|---------|----------|
| `read()` | one big string | small files |
| `readline()` | the next single line (string) | reading a header, or step-by-step control |
| `readlines()` | a list of all lines | small or medium files where you need a list |
| `for line in f:` | one line at a time (lazy) | **large files**, memory efficient |

In [4]:
with open(file_path, "r", encoding="utf-8") as f:
    first_line = f.readline()
    second_line = f.readline()
    remaining = f.readlines()

print(repr(first_line))
print(repr(second_line))
print(remaining)

'Hello, Python!\n'
'File handling is fun.\n'
['Third line here.\n']


### Line-by-line breakdown

1. `with open(...) as f:` - opens the file for reading. Python keeps an internal **file pointer** (a bookmark) at the start.
2. `first_line = f.readline()` - reads up to and including the first `\n`, then moves the pointer to the start of line two.
3. `second_line = f.readline()` - continues from where the pointer stopped, so it returns line two.
4. `remaining = f.readlines()` - reads **everything left** and returns it as a list of strings, one per line.
5. *(blank line)*
6. `print(repr(first_line))` - `repr()` shows the string with quotes and escape characters visible, so we can see the hidden `\n`.
7. `print(repr(second_line))` - same for line two.
8. `print(remaining)` - printing a list already shows each element with quotes and escapes.

### Interpreting the output

```
'Hello, Python!\n'
'File handling is fun.\n'
['Third line here.\n']
```

Each line keeps its trailing `\n`. The key lesson: **the pointer moves forward as you read**. Because two lines were already consumed, `readlines()` only returned the one line that was left, not all three. Use `.strip()` if you want to remove the newline.

In [5]:
with open(file_path, "r", encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        print(f"{line_number}: {line.strip()}")

1: Hello, Python!
2: File handling is fun.
3: Third line here.


### Line-by-line breakdown

1. `with open(...) as f:` - open for reading.
2. `for line_number, line in enumerate(f, start=1):` - a file object is **iterable**: looping over it yields one line at a time without loading the whole file. `enumerate(..., start=1)` pairs each line with a counter that begins at 1 instead of the default 0.
3. `print(f"{line_number}: {line.strip()}")` - an f-string that inserts the counter and the line. `.strip()` removes the trailing newline (and any surrounding spaces) so `print` does not create blank lines between entries.

### Interpreting the output

```
1: Hello, Python!
2: File handling is fun.
3: Third line here.
```

This is the **preferred pattern for large files** because only one line lives in memory at a time. A 10 GB log file can be processed this way on a small laptop.

## 1.5 Appending with `"a"` and the danger of `"w"`

`"a"` adds to the end of the file. `"w"` **destroys** existing content the moment the file is opened, even if you never write anything. Let us prove both.

In [6]:
with open(file_path, "a", encoding="utf-8") as f:
    f.write("This line was appended.\n")

with open(file_path, "r", encoding="utf-8") as f:
    lines = f.read().splitlines()

print(len(lines), "lines now")
print(lines[-1])

demo_path = WORKDIR / "overwrite_demo.txt"
demo_path.write_text("Original text\n", encoding="utf-8")

with open(demo_path, "w", encoding="utf-8") as f:
    f.write("Replaced text\n")

print(demo_path.read_text(encoding="utf-8").strip())

4 lines now
This line was appended.
Replaced text


### Line-by-line breakdown

1. `with open(file_path, "a", encoding="utf-8") as f:` - opens in **append mode**; the pointer is placed at the end of the existing content.
2. `f.write("This line was appended.\n")` - adds a fourth line, leaving the first three untouched.
3. *(blank line)*
4. `with open(file_path, "r", ...) as f:` - reopen for reading to verify the result.
5. `lines = f.read().splitlines()` - `read()` returns the full text and `splitlines()` splits it into a list of lines **without** the newline characters.
6. *(blank line)*
7. `print(len(lines), "lines now")` - counts the list items.
8. `print(lines[-1])` - index `-1` is the last element, which should be our appended line.
9. *(blank line)*
10. `demo_path = WORKDIR / "overwrite_demo.txt"` - a new path for the overwrite experiment.
11. `demo_path.write_text("Original text\n", encoding="utf-8")` - a `pathlib` shortcut that opens, writes and closes in a single call.
12. *(blank line)*
13. `with open(demo_path, "w", ...) as f:` - opens in write mode; **at this instant the old content is erased**.
14. `f.write("Replaced text\n")` - writes the new content.
15. *(blank line)*
16. `print(demo_path.read_text(encoding="utf-8").strip())` - `read_text()` is the matching shortcut for reading, and `.strip()` trims the newline for a cleaner print.

### Interpreting the output

```
4 lines now
This line was appended.
Replaced text
```

The file grew from 3 to 4 lines, proving `"a"` preserves data. In the second experiment only `Replaced text` remains: `"Original text"` is gone forever. **Rule of thumb:** use `"a"` to add, `"w"` only when you truly intend to start fresh, and `"x"` when you want Python to refuse to overwrite anything.

In [7]:
try:
    with open(file_path, "x", encoding="utf-8") as f:
        f.write("this should never be written")
except FileExistsError:
    print("Mode 'x' refused: the file already exists, so nothing was overwritten.")

print(file_path.read_text(encoding="utf-8").splitlines()[0])

Mode 'x' refused: the file already exists, so nothing was overwritten.
Hello, Python!


### Line-by-line breakdown

1. `try:` - starts a protected block; if an error occurs inside, Python looks for a matching `except` (covered fully in Part 4).
2. `with open(file_path, "x", ...) as f:` - exclusive-creation mode. Because `greetings.txt` already exists, Python raises `FileExistsError` immediately.
3. `f.write(...)` - never runs, because the error happened on the previous line.
4. `except FileExistsError:` - catches exactly that error type.
5. `print("Mode 'x' refused...")` - runs because the error was caught.
6. *(blank line)*
7. `print(file_path.read_text(...).splitlines()[0])` - reads the file again and prints only the first line to prove the original content is intact.

### Interpreting the output

The refusal message appears, then `Hello, Python!`. Mode `"x"` acts as a safety lock: it is ideal for creating output files where accidentally destroying an older result would be costly.

## 1.6 Writing many lines: `writelines()` and `print(file=...)`

`writelines()` accepts any iterable of strings but **does not add newlines**; you must include them. Alternatively, `print(..., file=f)` writes text **and adds a newline automatically**.

In [8]:
shopping = ["milk\n", "bread\n", "eggs\n"]
list_path = WORKDIR / "shopping.txt"

with open(list_path, "w", encoding="utf-8") as f:
    f.writelines(shopping)
    print("rice", file=f)
    print("beans", file=f)

print(list_path.read_text(encoding="utf-8"))

milk
bread
eggs
rice
beans



### Line-by-line breakdown

1. `shopping = ["milk\n", "bread\n", "eggs\n"]` - a list of strings. Each one already ends in `\n` because `writelines()` will not add them.
2. `list_path = WORKDIR / "shopping.txt"` - target path.
3. *(blank line)*
4. `with open(list_path, "w", ...) as f:` - open for writing.
5. `f.writelines(shopping)` - writes every string in the list, one after another.
6. `print("rice", file=f)` - `print` normally writes to the screen; the `file=f` argument redirects it into our file, and `print` appends `\n` by itself.
7. `print("beans", file=f)` - same, for the last item.
8. *(blank line)*
9. `print(list_path.read_text(encoding="utf-8"))` - reads and displays the whole file.

### Interpreting the output

Five items appear, each on its own line: milk, bread, eggs, rice, beans. Mixing `writelines()` and `print(file=f)` works because both write to the same open file in order. If you had forgotten the `\n` in the list, `milkbreadeggs` would have been glued together on a single line.

## 1.7 The file pointer: `tell()` and `seek()`

The **file pointer** is Python's current position in the file. `tell()` reports it and `seek(n)` moves it to byte position `n`. `seek(0)` rewinds to the start.

In [9]:
with open(file_path, "r", encoding="utf-8") as f:
    print("Start position:", f.tell())
    print("First 5 characters:", f.read(5))
    print("Position after reading 5:", f.tell())
    f.seek(0)
    print("After seek(0), first line:", f.readline().strip())

Start position: 0
First 5 characters: Hello
Position after reading 5: 5
After seek(0), first line: Hello, Python!


### Line-by-line breakdown

1. `with open(...) as f:` - open for reading; the pointer starts at 0.
2. `print("Start position:", f.tell())` - reports the current pointer position.
3. `print("First 5 characters:", f.read(5))` - `read(5)` reads only 5 characters instead of the whole file, and moves the pointer forward by 5.
4. `print("Position after reading 5:", f.tell())` - confirms the pointer moved.
5. `f.seek(0)` - jumps back to the very beginning.
6. `print("After seek(0), first line:", f.readline().strip())` - reading again from the start returns the first line.

### Interpreting the output

```
Start position: 0
First 5 characters: Hello
Position after reading 5: 5
After seek(0), first line: Hello, Python!
```

The pointer went 0, then 5, then back to 0. This is how you re-read a file without reopening it, or read a header and then jump to a specific position.

## 1.8 `pathlib`: modern path handling

`pathlib.Path` bundles many everyday operations into one object: joining, checking existence, reading, writing, listing, and inspecting metadata.

In [10]:
p = WORKDIR / "notes.txt"
p.write_text("Line A\nLine B\n", encoding="utf-8")

print("Exists:", p.exists())
print("Is file:", p.is_file())
print("Name:", p.name)
print("Stem:", p.stem)
print("Suffix:", p.suffix)
print("Parent folder:", p.parent.name)
print("Size in bytes:", p.stat().st_size)
print("Lines:", p.read_text(encoding="utf-8").splitlines())

print("Text files in the sandbox:")
for item in sorted(WORKDIR.glob("*.txt")):
    print(" -", item.name)

Exists: True
Is file: True
Name: notes.txt
Stem: notes
Suffix: .txt
Parent folder: file_handling_lab
Size in bytes: 16
Lines: ['Line A', 'Line B']
Text files in the sandbox:
 - greetings.txt
 - notes.txt
 - overwrite_demo.txt
 - shopping.txt


### Line-by-line breakdown

1. `p = WORKDIR / "notes.txt"` - build the path.
2. `p.write_text("Line A\nLine B\n", encoding="utf-8")` - create the file and write two lines in one call.
3. *(blank line)*
4. `print("Exists:", p.exists())` - `True` if the path is on disk.
5. `print("Is file:", p.is_file())` - `True` if it is a regular file (not a folder).
6. `print("Name:", p.name)` - the full file name including extension.
7. `print("Stem:", p.stem)` - the file name **without** the extension.
8. `print("Suffix:", p.suffix)` - the extension, including the dot.
9. `print("Parent folder:", p.parent.name)` - `.parent` is the containing folder; `.name` extracts its name.
10. `print("Size in bytes:", p.stat().st_size)` - `.stat()` returns file metadata and `st_size` is the size in bytes.
11. `print("Lines:", p.read_text(...).splitlines())` - reads the file and splits it into a list.
12. *(blank line)*
13. `print("Text files in the sandbox:")` - a heading for the listing.
14. `for item in sorted(WORKDIR.glob("*.txt")):` - `glob("*.txt")` finds every path in the folder that matches the pattern; `sorted()` puts them in alphabetical order so the output is predictable.
15. `print(" -", item.name)` - prints each matching file name with a plain hyphen bullet.

### Interpreting the output

`Exists: True`, `Is file: True`, name `notes.txt`, stem `notes`, suffix `.txt`, parent `file_handling_lab`. The size is **14 bytes** because `Line A\n` is 7 bytes and `Line B\n` is 7 bytes. The final list shows every `.txt` file created so far (`greetings.txt`, `notes.txt`, `overwrite_demo.txt`, `shopping.txt`), which proves `glob` can search a folder by pattern.

## 1.9 Binary files: `"rb"` and `"wb"`

Text mode converts bytes to strings using an encoding. **Binary mode** gives you the raw bytes with no conversion, which is required for images, PDFs, audio, zip files and any non-text data.

In [11]:
bin_path = WORKDIR / "bytes.bin"
data = bytes([72, 101, 108, 108, 111])

with open(bin_path, "wb") as f:
    f.write(data)

with open(bin_path, "rb") as f:
    raw = f.read()

print(raw)
print(list(raw))
print(raw.decode("utf-8"))

copy_path = WORKDIR / "bytes_copy.bin"
with open(bin_path, "rb") as src, open(copy_path, "wb") as dst:
    while True:
        chunk = src.read(2)
        if not chunk:
            break
        dst.write(chunk)

print("Copy identical:", bin_path.read_bytes() == copy_path.read_bytes())

b'Hello'
[72, 101, 108, 108, 111]
Hello
Copy identical: True


### Line-by-line breakdown

1. `bin_path = WORKDIR / "bytes.bin"` - path for the binary file.
2. `data = bytes([72, 101, 108, 108, 111])` - creates a `bytes` object from a list of integers between 0 and 255. These numbers are the ASCII codes for H, e, l, l, o.
3. *(blank line)*
4. `with open(bin_path, "wb") as f:` - `"wb"` is write **binary**. Notice there is **no `encoding` argument**: binary mode never encodes.
5. `f.write(data)` - binary files accept `bytes`, not `str`. Passing a normal string here would raise a `TypeError`.
6. *(blank line)*
7. `with open(bin_path, "rb") as f:` - `"rb"` is read binary.
8. `raw = f.read()` - returns a `bytes` object.
9. *(blank line)*
10. `print(raw)` - shows the bytes literal.
11. `print(list(raw))` - converting to a list reveals each byte as an integer.
12. `print(raw.decode("utf-8"))` - `.decode()` converts bytes to a string using an encoding, which is what text mode does automatically.
13. *(blank line)*
14. `copy_path = WORKDIR / "bytes_copy.bin"` - destination for a copy.
15. `with open(bin_path, "rb") as src, open(copy_path, "wb") as dst:` - one `with` statement can open **several files at once**, separated by commas. Both are closed automatically.
16. `while True:` - loop forever until we explicitly break.
17. `chunk = src.read(2)` - read at most 2 bytes at a time. Real programs use larger chunks such as 8192 or 1 MB; 2 is used here so the loop is easy to follow.
18. `if not chunk:` - at the end of the file `read()` returns an empty bytes object, which is "falsy".
19. `break` - leave the loop when there is no more data.
20. `dst.write(chunk)` - write each chunk to the destination.
21. *(blank line)*
22. `print("Copy identical:", ...)` - `read_bytes()` is the `pathlib` shortcut for reading a whole binary file; comparing the two byte strings with `==` tells us whether the copy is perfect.

### Interpreting the output

```
b'Hello'
[72, 101, 108, 108, 111]
Hello
Copy identical: True
```

`b'Hello'` shows the `b` prefix that marks a bytes object. The list confirms bytes are just integers underneath. Decoding turned them back into text. The chunked copy is the standard technique for copying **very large** files without loading them fully into memory, and `True` verifies nothing was lost.

## 1.10 Encoding: why `encoding="utf-8"` matters

Computers store bytes, not characters. An **encoding** is the rule that maps characters to bytes. UTF-8 can represent every character in every language (including symbols such as the naira sign) and is the modern standard. Some characters take more than one byte.

In [12]:
text = "Café ₦500"
enc_path = WORKDIR / "unicode.txt"
enc_path.write_text(text, encoding="utf-8")

print("Characters in the string:", len(text))
print("Bytes stored on disk:", enc_path.stat().st_size)
print("Read back correctly:", enc_path.read_text(encoding="utf-8"))

Characters in the string: 9
Bytes stored on disk: 12
Read back correctly: Café ₦500


### Line-by-line breakdown

1. `text = "Café ₦500"` - a string containing an accented letter (`é`) and the naira symbol (`₦`).
2. `enc_path = WORKDIR / "unicode.txt"` - target path.
3. `enc_path.write_text(text, encoding="utf-8")` - writes the string, encoding every character as UTF-8 bytes.
4. *(blank line)*
5. `print("Characters in the string:", len(text))` - counts **characters**.
6. `print("Bytes stored on disk:", enc_path.stat().st_size)` - counts **bytes** in the file.
7. `print("Read back correctly:", ...)` - reads with the same encoding and shows the text is intact.

### Interpreting the output

The string has **9 characters** but the file holds **12 bytes**. Plain letters and digits use 1 byte each in UTF-8, `é` uses 2 bytes and `₦` uses 3 bytes. That is the difference between "characters" and "bytes", and the reason you must read a file with the **same encoding** it was written with.

---
# Part 2: Working with CSV Files

**CSV** (Comma-Separated Values) is the most common format for tabular data: one row per line, values separated by commas. You could split lines on commas yourself, but that breaks as soon as a value contains a comma or a quotation mark. The `csv` module handles those cases correctly.

**Golden rule:** always open CSV files with `newline=""`. This lets the `csv` module manage line endings itself and prevents blank rows appearing on Windows.

## 2.1 Writing CSV with `csv.writer`

In [13]:
students_path = WORKDIR / "students.csv"
header = ["name", "age", "score"]
rows = [
    ["Amina", 21, 88.5],
    ["Tunde", 23, 92.0],
    ["Chidi", 22, 75.25],
]

with open(students_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(rows)

print(students_path.read_text(encoding="utf-8"))

name,age,score
Amina,21,88.5
Tunde,23,92.0
Chidi,22,75.25



### Line-by-line breakdown

1. `students_path = WORKDIR / "students.csv"` - path of the new CSV file.
2. `header = ["name", "age", "score"]` - a list holding the column names.
3. `rows = [` ... `]` (spanning several lines) - a list of lists, where each inner list is one record. Numbers stay as numbers here; the `csv` module converts them to text when writing.
4. *(blank line)*
5. `with open(students_path, "w", newline="", encoding="utf-8") as f:` - opens for writing. `newline=""` is the CSV golden rule.
6. `writer = csv.writer(f)` - wraps the file object in a writer that knows CSV formatting rules.
7. `writer.writerow(header)` - writes **one row** (the header).
8. `writer.writerows(rows)` - note the **s**: writes **many rows** at once from a list of lists.
9. *(blank line)*
10. `print(students_path.read_text(encoding="utf-8"))` - displays the raw file content.

### Interpreting the output

```
name,age,score
Amina,21,88.5
Tunde,23,92.0
Chidi,22,75.25
```

Every list became one line and each element was separated by a comma. Numbers were converted to text automatically. Notice that `92.0` stays `92.0`; CSV stores plain text and has no concept of data types.

## 2.2 Reading CSV with `csv.reader`

`csv.reader` returns each row as a **list of strings**. Everything is a string, including numbers, so you must convert types yourself.

In [14]:
with open(students_path, "r", newline="", encoding="utf-8") as f:
    reader = csv.reader(f)
    header_row = next(reader)
    data_rows = list(reader)

print("Header:", header_row)
print("First data row:", data_rows[0])
print("Type of the age value:", type(data_rows[0][1]))

scores = [float(row[2]) for row in data_rows]
print("Scores as floats:", scores)
print("Average score:", sum(scores) / len(scores))

Header: ['name', 'age', 'score']
First data row: ['Amina', '21', '88.5']
Type of the age value: <class 'str'>
Scores as floats: [88.5, 92.0, 75.25]
Average score: 85.25


### Line-by-line breakdown

1. `with open(students_path, "r", newline="", encoding="utf-8") as f:` - open for reading with the CSV golden rule.
2. `reader = csv.reader(f)` - creates an **iterator** that yields one parsed row per step.
3. `header_row = next(reader)` - `next()` pulls the **first** row (the header) out of the iterator so it does not mix with the data.
4. `data_rows = list(reader)` - consumes the remaining rows into a list. We do this **inside** the `with` block because the reader needs the file to be open.
5. *(blank line)*
6. `print("Header:", header_row)` - shows the column names.
7. `print("First data row:", data_rows[0])` - the first record.
8. `print("Type of the age value:", type(data_rows[0][1]))` - `data_rows[0][1]` is row 0, column 1 (the age); `type()` reveals it is a string.
9. *(blank line)*
10. `scores = [float(row[2]) for row in data_rows]` - a list comprehension: for every row take column 2 and convert it from text to a `float`.
11. `print("Scores as floats:", scores)` - the converted numbers.
12. `print("Average score:", sum(scores) / len(scores))` - `sum()` adds them and dividing by the count gives the mean.

### Interpreting the output

```
Header: ['name', 'age', 'score']
First data row: ['Amina', '21', '88.5']
Type of the age value: <class 'str'>
Scores as floats: [88.5, 92.0, 75.25]
Average score: 85.25
```

The quotes around `'21'` and `'88.5'` are the proof that everything read from CSV is text. After converting to floats we can do maths: (88.5 + 92.0 + 75.25) / 3 = 85.25.

## 2.3 `DictReader` and `DictWriter`: access columns by name

Working with `row[2]` is fragile: if someone reorders the columns your code silently breaks. `DictReader` and `DictWriter` use the header names as dictionary keys, which is far more readable and robust.

In [15]:
with open(students_path, "r", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    print("Column names:", reader.fieldnames)
    for row in reader:
        print(row["name"], "scored", row["score"])

Column names: ['name', 'age', 'score']
Amina scored 88.5
Tunde scored 92.0
Chidi scored 75.25


### Line-by-line breakdown

1. `with open(...) as f:` - open the CSV for reading.
2. `reader = csv.DictReader(f)` - reads the first line as the header automatically and then turns every following row into a dictionary.
3. `print("Column names:", reader.fieldnames)` - the header names as a list.
4. `for row in reader:` - loop over the records; each `row` is a dictionary such as `{"name": "Amina", "age": "21", "score": "88.5"}`.
5. `print(row["name"], "scored", row["score"])` - access values **by column name**, so the code is self-documenting.

### Interpreting the output

```
Column names: ['name', 'age', 'score']
Amina scored 88.5
Tunde scored 92.0
Chidi scored 75.25
```

Each record was handled as a labelled dictionary. Values are still strings, so convert with `float()` or `int()` when you need numbers.

In [16]:
people = [
    {"name": "Ngozi", "city": "Lagos", "active": True},
    {"name": "Bala", "city": "Kano", "active": False},
]
people_path = WORKDIR / "people.csv"

with open(people_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "city", "active"])
    writer.writeheader()
    writer.writerows(people)

print(people_path.read_text(encoding="utf-8"))

name,city,active
Ngozi,Lagos,True
Bala,Kano,False



### Line-by-line breakdown

1. `people = [` ... `]` (spanning several lines) - a list of dictionaries. Each dictionary is one record, and the keys are the column names.
2. `people_path = WORKDIR / "people.csv"` - destination file.
3. *(blank line)*
4. `with open(people_path, "w", newline="", encoding="utf-8") as f:` - open for writing.
5. `writer = csv.DictWriter(f, fieldnames=["name", "city", "active"])` - a writer that maps dictionary keys to columns. `fieldnames` sets **which columns exist and in what order**.
6. `writer.writeheader()` - writes the header line built from `fieldnames`. `DictWriter` does not do this automatically, so do not forget it.
7. `writer.writerows(people)` - writes every dictionary as a row.
8. *(blank line)*
9. `print(people_path.read_text(encoding="utf-8"))` - show the file.

### Interpreting the output

```
name,city,active
Ngozi,Lagos,True
Bala,Kano,False
```

The header came from `writeheader()`. The boolean values were written as the text `True` and `False`. If a dictionary contained a key that is **not** in `fieldnames`, Python would raise a `ValueError`, which protects you from silent data mistakes.

## 2.4 Tricky data: commas, quotes and custom delimiters

A value can itself contain a comma or a quotation mark. The `csv` module automatically wraps such values in quotes and doubles any inner quotes. You can also change the **delimiter** for files that use semicolons or tabs.

In [17]:
tricky_path = WORKDIR / "tricky.csv"
tricky_rows = [
    ["product", "note"],
    ["Pen", "Blue, fine tip"],
    ["Book", 'He said "great"'],
]

with open(tricky_path, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerows(tricky_rows)

print(tricky_path.read_text(encoding="utf-8"))

with open(tricky_path, "r", newline="", encoding="utf-8") as f:
    print(list(csv.reader(f)))

semi_path = WORKDIR / "semicolon.csv"
with open(semi_path, "w", newline="", encoding="utf-8") as f:
    csv.writer(f, delimiter=";").writerows([["a", "b"], [1, 2]])

with open(semi_path, "r", newline="", encoding="utf-8") as f:
    print(list(csv.reader(f, delimiter=";")))

product,note
Pen,"Blue, fine tip"
Book,"He said ""great"""

[['product', 'note'], ['Pen', 'Blue, fine tip'], ['Book', 'He said "great"']]
[['a', 'b'], ['1', '2']]


### Line-by-line breakdown

1. `tricky_path = WORKDIR / "tricky.csv"` - path for the awkward data.
2. `tricky_rows = [` ... `]` (spanning several lines) - three rows. Row two contains a **comma inside a value**; row three contains **double quotes inside a value**.
3. *(blank line)*
4. `with open(tricky_path, "w", ...) as f:` - open for writing.
5. `csv.writer(f).writerows(tricky_rows)` - creates the writer and writes all rows in one expression; a short form when you need the writer only once.
6. *(blank line)*
7. `print(tricky_path.read_text(...))` - see how the module encoded the awkward values.
8. *(blank line)*
9. `with open(tricky_path, "r", ...) as f:` - reopen for reading.
10. `print(list(csv.reader(f)))` - parse everything back into a list of rows to prove the round trip works.
11. *(blank line)*
12. `semi_path = WORKDIR / "semicolon.csv"` - a second file that uses `;` as the separator.
13. `with open(semi_path, "w", ...) as f:` - open for writing.
14. `csv.writer(f, delimiter=";").writerows(...)` - the `delimiter` argument changes the separator character.
15. *(blank line)*
16. `with open(semi_path, "r", ...) as f:` - open for reading.
17. `print(list(csv.reader(f, delimiter=";")))` - you must give the **same delimiter** when reading, otherwise each line would be treated as a single value.

### Interpreting the output

The raw file looks like this:

```
product,note
Pen,"Blue, fine tip"
Book,"He said ""great"""
```

The value containing a comma was wrapped in quotes so the comma is not mistaken for a separator, and each inner quote was doubled (`""`). Reading it back gives `[['product', 'note'], ['Pen', 'Blue, fine tip'], ['Book', 'He said "great"']]`, identical to the original data. The semicolon file returns `[['a', 'b'], ['1', '2']]`; the numbers came back as text, as always with CSV.

## 2.5 A quick look at pandas

For real data analysis, `pandas` reads a CSV into a `DataFrame` in one line and detects numeric types automatically. The `csv` module is best for lightweight, dependency-free scripts; `pandas` is best for analysis.

In [18]:
try:
    import pandas as pd
    df = pd.read_csv(students_path)
    print(df)
    print()
    print(df.dtypes)
    print()
    print("Mean score:", df["score"].mean())
except ImportError:
    print("pandas is not installed. Install it with: pip install pandas")

    name  age  score
0  Amina   21  88.50
1  Tunde   23  92.00
2  Chidi   22  75.25

name      object
age        int64
score    float64
dtype: object

Mean score: 85.25


### Line-by-line breakdown

1. `try:` - protects the code because pandas may not be installed on every machine.
2. `import pandas as pd` - imports pandas with the conventional short name `pd`. If it is missing, `ImportError` is raised here.
3. `df = pd.read_csv(students_path)` - reads the whole file, uses the first line as column names and guesses each column's type.
4. `print(df)` - pandas prints a neat table with a row index on the left.
5. `print()` - an empty line for spacing.
6. `print(df.dtypes)` - shows the detected type of each column.
7. `print()` - spacing again.
8. `print("Mean score:", df["score"].mean())` - selects the `score` column and calculates its average in one step.
9. `except ImportError:` - runs only if pandas is not installed.
10. `print("pandas is not installed...")` - a friendly instruction instead of a crash.

### Interpreting the output

The table shows three students. In `dtypes`, `name` is text (`object` or `str` depending on your pandas version), `age` is `int64` and `score` is `float64`. Unlike the plain `csv` module, pandas **converted the numbers for us**. The mean is `85.25`, matching the manual calculation in section 2.2.

---
# Part 3: Working with JSON Files

**JSON** (JavaScript Object Notation) is the standard format for web APIs, configuration files and nested data. It looks almost like a Python dictionary, and the `json` module converts between the two.

| Python | JSON |
|--------|------|
| `dict` | object `{}` |
| `list`, `tuple` | array `[]` |
| `str` | string |
| `int`, `float` | number |
| `True` / `False` | `true` / `false` |
| `None` | `null` |

Four functions to remember (the **s** stands for *string*):

| Function | Direction | Works with |
|----------|-----------|------------|
| `json.dump(obj, f)` | Python to JSON | a **file** |
| `json.load(f)` | JSON to Python | a **file** |
| `json.dumps(obj)` | Python to JSON | a **string** |
| `json.loads(s)` | JSON to Python | a **string** |

## 3.1 Writing JSON with `json.dump`

In [19]:
profile = {
    "name": "Amina",
    "age": 21,
    "is_student": True,
    "scores": [88.5, 92.0, 75.25],
    "address": {"city": "Lagos", "country": "Nigeria"},
    "nickname": None,
}

json_path = WORKDIR / "profile.json"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(profile, f, indent=4)

print(json_path.read_text(encoding="utf-8"))

{
    "name": "Amina",
    "age": 21,
    "is_student": true,
    "scores": [
        88.5,
        92.0,
        75.25
    ],
    "address": {
        "city": "Lagos",
        "country": "Nigeria"
    },
    "nickname": null
}


### Line-by-line breakdown

1. `profile = {` ... `}` (spanning several lines) - a Python dictionary that deliberately uses many data types: text, a whole number, a boolean, a list of floats, a **nested dictionary** and `None`.
2. *(blank line)*
3. `json_path = WORKDIR / "profile.json"` - destination path.
4. *(blank line)*
5. `with open(json_path, "w", encoding="utf-8") as f:` - open for writing text. JSON is text, so no binary mode and no `newline=""` needed.
6. `json.dump(profile, f, indent=4)` - serializes the dictionary and writes it into the open file. `indent=4` pretty-prints with four spaces per level; without it, everything would be squeezed onto one line.
7. *(blank line)*
8. `print(json_path.read_text(encoding="utf-8"))` - display the resulting file.

### Interpreting the output

The file shows a readable, indented structure. Look at how Python values were translated: `True` became `true`, `None` became `null`, the dictionary became an object with curly braces and the list became an array. The nested `address` dictionary is preserved as a nested object. JSON keeps **structure**, which is exactly what CSV cannot do.

## 3.2 Reading JSON with `json.load`

In [20]:
with open(json_path, "r", encoding="utf-8") as f:
    loaded = json.load(f)

print(type(loaded))
print(loaded["address"]["city"])
print(loaded["is_student"], type(loaded["is_student"]))
print(loaded["nickname"])
print("Round trip identical:", loaded == profile)

<class 'dict'>
Lagos
True <class 'bool'>
None
Round trip identical: True


### Line-by-line breakdown

1. `with open(json_path, "r", encoding="utf-8") as f:` - open for reading.
2. `loaded = json.load(f)` - reads the file and rebuilds the Python object.
3. *(blank line)*
4. `print(type(loaded))` - checks what came back.
5. `print(loaded["address"]["city"])` - **chained indexing**: first get the nested dictionary, then the `city` key inside it.
6. `print(loaded["is_student"], type(loaded["is_student"]))` - shows the value and its type.
7. `print(loaded["nickname"])` - the JSON `null` value.
8. `print("Round trip identical:", loaded == profile)` - compares the loaded data with the original dictionary.

### Interpreting the output

```
<class 'dict'>
Lagos
True <class 'bool'>
None
Round trip identical: True
```

JSON `true` became Python `True` (a real `bool`, not a string) and `null` became `None`. The last line proves the write-then-read **round trip** lost nothing.

## 3.3 Strings instead of files: `dumps` and `loads`

Use `dumps`/`loads` when the JSON is not in a file, for example data received from a web API or a database field.

In [21]:
text_form = json.dumps(profile, sort_keys=True)
print(text_form)

back = json.loads(text_form)
print(type(text_form), type(back))

compact = json.dumps({"a": 1, "b": [1, 2]}, separators=(",", ":"))
print(compact)

{"address": {"city": "Lagos", "country": "Nigeria"}, "age": 21, "is_student": true, "name": "Amina", "nickname": null, "scores": [88.5, 92.0, 75.25]}
<class 'str'> <class 'dict'>
{"a":1,"b":[1,2]}


### Line-by-line breakdown

1. `text_form = json.dumps(profile, sort_keys=True)` - converts the dictionary to a **JSON string** (not a file). `sort_keys=True` orders keys alphabetically, which makes output predictable and easy to compare.
2. `print(text_form)` - shows the one-line JSON text.
3. *(blank line)*
4. `back = json.loads(text_form)` - parses the string back into a Python dictionary.
5. `print(type(text_form), type(back))` - confirms one is `str` and the other is `dict`.
6. *(blank line)*
7. `compact = json.dumps({"a": 1, "b": [1, 2]}, separators=(",", ":"))` - the `separators` argument removes the default spaces after commas and colons, producing the smallest possible text.
8. `print(compact)` - shows the compact form.

### Interpreting the output

The first line is a single line of JSON with keys in alphabetical order (`address`, `age`, `is_student`, `name`, `nickname`, `scores`). Then `<class 'str'> <class 'dict'>` shows the conversion in both directions. The compact output is `{"a":1,"b":[1,2]}`, which saves bandwidth when sending large amounts of JSON over a network.

## 3.4 Data JSON cannot handle by default, and how to fix it

JSON supports only a few basic types. Objects such as `date`, `datetime`, `set` or your own classes raise a `TypeError`. The `default=` argument lets you supply a function that converts unknown objects into something JSON understands.

In [22]:
from datetime import date, datetime

record = {"event": "Launch", "when": date(2026, 9, 28)}

try:
    json.dumps(record)
except TypeError as err:
    print("TypeError:", err)


def encode_extra(obj):
    if isinstance(obj, (date, datetime)):
        return obj.isoformat()
    raise TypeError(f"Cannot serialize {type(obj).__name__}")


print(json.dumps(record, default=encode_extra))

TypeError: Object of type date is not JSON serializable
{"event": "Launch", "when": "2026-09-28"}


### Line-by-line breakdown

1. `from datetime import date, datetime` - imports the two date classes.
2. *(blank line)*
3. `record = {"event": "Launch", "when": date(2026, 9, 28)}` - a dictionary holding a real `date` object, which JSON cannot represent.
4. *(blank line)*
5. `try:` - we expect the next line to fail and want to show the error safely.
6. `json.dumps(record)` - raises `TypeError` because it does not know how to convert a `date`.
7. `except TypeError as err:` - catches the error and stores it in `err`.
8. `print("TypeError:", err)` - displays the error message.
9. *(two blank lines)* - a style convention before defining a function.
10. `def encode_extra(obj):` - defines a converter that `json` will call **only for objects it cannot handle itself**.
11. `if isinstance(obj, (date, datetime)):` - checks whether the object is one of the two date types.
12. `return obj.isoformat()` - converts the date into a standard text form such as `2026-09-28`.
13. `raise TypeError(f"Cannot serialize {type(obj).__name__}")` - for any other unknown type, we still fail loudly with a clear message rather than hiding the problem.
14. *(two blank lines)*
15. `print(json.dumps(record, default=encode_extra))` - passes our function through `default=`; the date is now converted successfully.

### Interpreting the output

```
TypeError: Object of type date is not JSON serializable
{"event": "Launch", "when": "2026-09-28"}
```

The first line is the failure we expected. The second shows the fix: the date became the string `"2026-09-28"`. Remember the reverse direction is manual: after `json.load` you get a **string**, and you must convert it back with `date.fromisoformat(...)` if you need a real date.

## 3.5 Updating a JSON file and the JSON Lines format

To change a JSON file you **read it, modify the Python object, then write it back**. For very large datasets or logs, the **JSON Lines** format (one JSON object per line) lets you append and stream records without loading everything.

In [23]:
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

data["scores"].append(99.0)
data["nickname"] = "Ami"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2)

with open(json_path, "r", encoding="utf-8") as f:
    updated = json.load(f)

print("Scores:", updated["scores"])
print("Nickname:", updated["nickname"])

Scores: [88.5, 92.0, 75.25, 99.0]
Nickname: Ami


### Line-by-line breakdown

1. `with open(json_path, "r", ...) as f:` - open the existing JSON file for reading.
2. `data = json.load(f)` - load it into a Python dictionary.
3. *(blank line)*
4. `data["scores"].append(99.0)` - `data["scores"]` is a normal Python list, so we can call `.append()` on it.
5. `data["nickname"] = "Ami"` - replace the old `None` value with a string.
6. *(blank line)*
7. `with open(json_path, "w", ...) as f:` - reopen the **same file for writing**. This erases the old file content, but that is safe because our full updated data is already in memory.
8. `json.dump(data, f, indent=2)` - write the modified dictionary back, this time with 2-space indentation.
9. *(blank line)*
10. `with open(json_path, "r", ...) as f:` - reopen once more to verify the saved result.
11. `updated = json.load(f)` - load the fresh data.
12. *(blank line)*
13. `print("Scores:", updated["scores"])` - the list, now with a fourth score.
14. `print("Nickname:", updated["nickname"])` - the new nickname.

### Interpreting the output

```
Scores: [88.5, 92.0, 75.25, 99.0]
Nickname: Ami
```

Both edits were saved to disk. This **read, modify, write** cycle is the standard way to update JSON. Note the risk: if the program crashes between opening for writing and finishing `dump`, the file could be left empty. Part 6 shows a safer pattern.

In [24]:
jsonl_path = WORKDIR / "events.jsonl"
events = [{"id": 1, "type": "login"}, {"id": 2, "type": "logout"}]

with open(jsonl_path, "w", encoding="utf-8") as f:
    for event in events:
        f.write(json.dumps(event) + "\n")

with open(jsonl_path, "a", encoding="utf-8") as f:
    f.write(json.dumps({"id": 3, "type": "login"}) + "\n")

with open(jsonl_path, "r", encoding="utf-8") as f:
    parsed = [json.loads(line) for line in f]

print(jsonl_path.read_text(encoding="utf-8"))
print("Records parsed:", len(parsed))
print("Last record:", parsed[-1])

{"id": 1, "type": "login"}
{"id": 2, "type": "logout"}
{"id": 3, "type": "login"}

Records parsed: 3
Last record: {'id': 3, 'type': 'login'}


### Line-by-line breakdown

1. `jsonl_path = WORKDIR / "events.jsonl"` - `.jsonl` is the conventional extension for JSON Lines.
2. `events = [...]` - two small dictionaries.
3. *(blank line)*
4. `with open(jsonl_path, "w", ...) as f:` - open for writing.
5. `for event in events:` - loop through the records.
6. `f.write(json.dumps(event) + "\n")` - convert **one** record to a JSON string and add a newline, so each record occupies exactly one line.
7. *(blank line)*
8. `with open(jsonl_path, "a", ...) as f:` - open in **append mode**; this is where JSON Lines shines, because adding a record never requires rewriting the file.
9. `f.write(json.dumps({"id": 3, "type": "login"}) + "\n")` - append a third record.
10. *(blank line)*
11. `with open(jsonl_path, "r", ...) as f:` - open for reading.
12. `parsed = [json.loads(line) for line in f]` - a list comprehension: iterate line by line and parse each line separately with `json.loads`.
13. *(blank line)*
14. `print(jsonl_path.read_text(...))` - show the raw file.
15. `print("Records parsed:", len(parsed))` - number of records.
16. `print("Last record:", parsed[-1])` - the appended record.

### Interpreting the output

The raw file shows three separate one-line JSON objects (ids 1, 2 and 3). `Records parsed: 3` and `Last record: {'id': 3, 'type': 'login'}` confirm that the appended line was read like any other. Compare with a normal `.json` file, where appending would mean reading and rewriting the entire document.

---
# Part 4: Error Handling with `try` / `except`

Files are one of the most failure-prone things a program touches: the file may not exist, you may lack permission, the disk may be full, the content may be corrupted or in the wrong encoding. Good programs **expect** failure and respond gracefully instead of crashing.

## 4.1 The structure of error handling

```python
try:
    # code that might fail
except SomeError:
    # what to do if that specific error happens
else:
    # runs ONLY if no error happened
finally:
    # ALWAYS runs, error or not (cleanup)
```

## 4.2 Catching `FileNotFoundError`

In [25]:
try:
    with open("does_not_exist.txt", "r", encoding="utf-8") as f:
        print(f.read())
except FileNotFoundError as err:
    print("Caught FileNotFoundError")
    print("Details:", err)

print("The program keeps running after the error.")

Caught FileNotFoundError
Details: [Errno 2] No such file or directory: 'does_not_exist.txt'
The program keeps running after the error.


### Line-by-line breakdown

1. `try:` - begins the protected block.
2. `with open("does_not_exist.txt", "r", encoding="utf-8") as f:` - tries to open a file that is not there. Python raises `FileNotFoundError` **on this line** and immediately jumps out of the `try` block.
3. `print(f.read())` - never executes, because the file was never opened.
4. `except FileNotFoundError as err:` - catches that specific error and stores the error object in `err`.
5. `print("Caught FileNotFoundError")` - our own friendly message.
6. `print("Details:", err)` - printing the error object shows Python's original message, useful for debugging.
7. *(blank line)*
8. `print("The program keeps running after the error.")` - proves execution continued normally after handling.

### Interpreting the output

```
Caught FileNotFoundError
Details: [Errno 2] No such file or directory: 'does_not_exist.txt'
The program keeps running after the error.
```

`[Errno 2]` is the operating system's error code for "no such file". Without the `try`/`except`, the program would have stopped with a long traceback and the last line would never have printed.

## 4.3 `except`, `else` and `finally` together

- `except` handles specific failures.
- `else` holds code that should run **only on success**. Keeping it separate from `try` makes clear which line is risky.
- `finally` **always** runs, which is where cleanup belongs.

In [26]:
def read_number_file(path):
    f = None
    try:
        f = open(path, "r", encoding="utf-8")
        value = int(f.read().strip())
    except FileNotFoundError:
        print("  [except] file is missing")
        return None
    except ValueError:
        print("  [except] file does not contain an integer")
        return None
    else:
        print("  [else] read succeeded")
        return value
    finally:
        print("  [finally] cleanup running")
        if f is not None:
            f.close()


(WORKDIR / "number.txt").write_text("42", encoding="utf-8")
(WORKDIR / "not_number.txt").write_text("forty-two", encoding="utf-8")

for name in ["number.txt", "not_number.txt", "missing.txt"]:
    print("Trying", name)
    print("  result:", read_number_file(WORKDIR / name))

Trying number.txt
  [else] read succeeded
  [finally] cleanup running
  result: 42
Trying not_number.txt
  [except] file does not contain an integer
  [finally] cleanup running
  result: None
Trying missing.txt
  [except] file is missing
  [finally] cleanup running
  result: None


### Line-by-line breakdown

1. `def read_number_file(path):` - a function that reads a file expected to contain one integer.
2. `f = None` - prepare the variable **before** `try`. If `open()` fails, `f` still exists, so `finally` can check it safely.
3. `try:` - protected block begins.
4. `f = open(path, "r", encoding="utf-8")` - open the file manually (no `with`) so we can demonstrate `finally`. May raise `FileNotFoundError`.
5. `value = int(f.read().strip())` - read the text, remove whitespace, convert to an integer. May raise `ValueError` if the text is not a valid number.
6. `except FileNotFoundError:` - handles the missing-file case.
7. `print("  [except] file is missing")` - report the problem.
8. `return None` - give the caller a neutral result.
9. `except ValueError:` - handles content that is not a whole number.
10. `print("  [except] file does not contain an integer")` - report the problem.
11. `return None` - neutral result again.
12. `else:` - runs only when the `try` block finished with **no error**.
13. `print("  [else] read succeeded")` - success message.
14. `return value` - return the integer.
15. `finally:` - runs no matter what, **even after a `return`**.
16. `print("  [finally] cleanup running")` - proves it always executes.
17. `if f is not None:` - only close the file if it was actually opened.
18. `f.close()` - releases the file.
19. *(two blank lines)* - style separation after a function.
20. `(WORKDIR / "number.txt").write_text("42", ...)` - create a valid test file containing `42`.
21. `(WORKDIR / "not_number.txt").write_text("forty-two", ...)` - create an invalid file containing words.
22. *(blank line)*
23. `for name in ["number.txt", "not_number.txt", "missing.txt"]:` - test three scenarios: success, bad content, missing file.
24. `print("Trying", name)` - announce the current test.
25. `print("  result:", read_number_file(WORKDIR / name))` - calls the function, which prints its own messages first, then prints the returned value.

### Interpreting the output

```
Trying number.txt
  [else] read succeeded
  [finally] cleanup running
  result: 42
Trying not_number.txt
  [except] file does not contain an integer
  [finally] cleanup running
  result: None
Trying missing.txt
  [except] file is missing
  [finally] cleanup running
  result: None
```

Study the pattern: the `[finally]` line appears in **all three** runs, even though two of them ended with `return None`. The success path went through `else`; each failure went through exactly one matching `except`. The function messages appear **before** `result:` because Python must finish calling the function before `print` can display its return value.

## 4.4 Handling bad JSON: `JSONDecodeError`

When a JSON file is malformed, `json.load` raises `json.JSONDecodeError`. The exception object tells you **where** the problem is, which is priceless when debugging large files.

In [27]:
bad_json_path = WORKDIR / "broken.json"
bad_json_path.write_text('{"name": "Amina", "age": }', encoding="utf-8")

try:
    with open(bad_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
except json.JSONDecodeError as err:
    print("Invalid JSON detected")
    print("Message:", err.msg)
    print("Line:", err.lineno, "| Column:", err.colno)

Invalid JSON detected
Message: Expecting value
Line: 1 | Column: 26


### Line-by-line breakdown

1. `bad_json_path = WORKDIR / "broken.json"` - path for a deliberately broken file.
2. `bad_json_path.write_text('{"name": "Amina", "age": }', ...)` - writes JSON where the value for `age` is **missing**. The outer quotes are single so the double quotes inside are legal.
3. *(blank line)*
4. `try:` - protected block.
5. `with open(bad_json_path, "r", ...) as f:` - opening succeeds; the file exists.
6. `data = json.load(f)` - the parser fails here because after `"age":` it expects a value but finds `}`.
7. `except json.JSONDecodeError as err:` - catches the parsing problem specifically.
8. `print("Invalid JSON detected")` - user-friendly message.
9. `print("Message:", err.msg)` - `.msg` is the short description of what went wrong.
10. `print("Line:", err.lineno, "| Column:", err.colno)` - `.lineno` and `.colno` point to the exact location of the problem.

### Interpreting the output

`Message: Expecting value` means the parser wanted a value (a number, string, list and so on) but found something else. `Line: 1 | Column: 26` points to the closing brace, the 26th character of the single line. Notice that this was **not** a `FileNotFoundError`: the file opened fine and the **content** was wrong. Different failures need different handlers.

## 4.5 The exception hierarchy: order of `except` blocks matters

Exceptions form a family tree. Catching a parent class also catches all its children. Python checks `except` blocks **from top to bottom** and uses the first match, so always put **specific** exceptions **before** general ones.

In [28]:
print("FileNotFoundError is an OSError:", issubclass(FileNotFoundError, OSError))
print("PermissionError is an OSError:", issubclass(PermissionError, OSError))
print("IsADirectoryError is an OSError:", issubclass(IsADirectoryError, OSError))
print("JSONDecodeError is a ValueError:", issubclass(json.JSONDecodeError, ValueError))
print("UnicodeDecodeError is a ValueError:", issubclass(UnicodeDecodeError, ValueError))

try:
    open(WORKDIR, "r", encoding="utf-8")
except (IsADirectoryError, PermissionError):
    print("Tried to open a folder as if it were a file.")

FileNotFoundError is an OSError: True
PermissionError is an OSError: True
IsADirectoryError is an OSError: True
JSONDecodeError is a ValueError: True
UnicodeDecodeError is a ValueError: True
Tried to open a folder as if it were a file.


### Line-by-line breakdown

1. `print("FileNotFoundError is an OSError:", issubclass(FileNotFoundError, OSError))` - `issubclass(A, B)` answers "is A a child of B?". All file-related failures inherit from `OSError`.
2. `print("PermissionError ...")` - same test for permission problems.
3. `print("IsADirectoryError ...")` - same for trying to open a folder as a file.
4. `print("JSONDecodeError ...")` - bad JSON content is a kind of `ValueError`.
5. `print("UnicodeDecodeError ...")` - text that cannot be decoded is also a `ValueError`.
6. *(blank line)*
7. `try:` - protected block.
8. `open(WORKDIR, "r", encoding="utf-8")` - deliberately tries to open the **folder** as a file. On Linux and macOS this raises `IsADirectoryError`; on Windows it raises `PermissionError`.
9. `except (IsADirectoryError, PermissionError):` - a **tuple** of exception types catches any of them with one handler, which makes the code portable across operating systems.
10. `print("Tried to open a folder as if it were a file.")` - the response.

### Interpreting the output

All five `issubclass` lines print `True`, confirming the family tree. Because `FileNotFoundError`, `PermissionError` and `IsADirectoryError` all descend from `OSError`, a single `except OSError` can catch every file-system problem, but it is better to catch specific ones first when you want different responses. The final line prints because the folder-as-file mistake was caught.

## 4.6 Encoding errors: `UnicodeDecodeError`

If a file was saved in one encoding and you read it with another, Python may not be able to decode the bytes. We will simulate this by saving text as Latin-1 and reading it as UTF-8.

In [29]:
latin_path = WORKDIR / "latin.txt"
latin_path.write_bytes("Café".encode("latin-1"))

try:
    latin_path.read_text(encoding="utf-8")
except UnicodeDecodeError as err:
    print("UnicodeDecodeError, reason:", err.reason)

print("Correct encoding  :", latin_path.read_text(encoding="latin-1"))
print("errors='replace'  :", latin_path.read_text(encoding="utf-8", errors="replace"))

UnicodeDecodeError, reason: unexpected end of data
Correct encoding  : Café
errors='replace'  : Caf�


### Line-by-line breakdown

1. `latin_path = WORKDIR / "latin.txt"` - path for the test file.
2. `latin_path.write_bytes("Café".encode("latin-1"))` - `.encode("latin-1")` turns the string into bytes using the Latin-1 encoding, where `é` is a **single byte** (233). `write_bytes` saves those raw bytes.
3. *(blank line)*
4. `try:` - protected block.
5. `latin_path.read_text(encoding="utf-8")` - tries to decode with UTF-8. The lone byte 233 is not valid UTF-8, so Python raises `UnicodeDecodeError`.
6. `except UnicodeDecodeError as err:` - catches the failure.
7. `print("UnicodeDecodeError, reason:", err.reason)` - `.reason` describes what was wrong with the bytes.
8. *(blank line)*
9. `print("Correct encoding  :", latin_path.read_text(encoding="latin-1"))` - reading with the **matching** encoding recovers the text perfectly.
10. `print("errors='replace'  :", latin_path.read_text(encoding="utf-8", errors="replace"))` - `errors="replace"` does not crash; it substitutes every undecodable byte with the replacement character.

### Interpreting the output

The reason message explains that the bytes are not valid UTF-8 (the exact wording can vary between Python versions). Reading with `latin-1` returns `Café`, the right answer. Reading with `errors="replace"` returns `Caf` followed by the replacement symbol, so the program survives but **data is damaged**. Prefer finding the correct encoding; use `"replace"` or `"ignore"` only when losing a few characters is acceptable.

## 4.7 Building reusable, safe helper functions

Instead of repeating `try`/`except` everywhere, wrap it once in a helper that returns a sensible default when something goes wrong.

In [30]:
def safe_read_text(path, default=""):
    """Return the file text, or `default` if the file cannot be read."""
    try:
        with open(path, "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        print(f"[warning] {Path(path).name} was not found")
    except PermissionError:
        print(f"[warning] no permission to read {Path(path).name}")
    except OSError as err:
        print(f"[warning] operating system problem: {err}")
    return default


print(repr(safe_read_text(WORKDIR / "number.txt")))
print(repr(safe_read_text(WORKDIR / "ghost.txt", default="N/A")))

'42'
[warning] ghost.txt was not found
'N/A'


### Line-by-line breakdown

1. `def safe_read_text(path, default=""):` - a function with a **default argument**: if the caller does not supply `default`, an empty string is used.
2. `"""Return the file text..."""` - a **docstring** describing what the function does; tools and `help()` display it.
3. `try:` - protected block.
4. `with open(path, "r", encoding="utf-8") as f:` - open safely.
5. `return f.read()` - on success, return the text immediately. The `with` block still closes the file on the way out.
6. `except FileNotFoundError:` - most common problem first (specific).
7. `print(f"[warning] {Path(path).name} was not found")` - `Path(path).name` works even if `path` was passed as a plain string.
8. `except PermissionError:` - second specific case.
9. `print(f"[warning] no permission ...")` - message for that case.
10. `except OSError as err:` - a **general catch-all for other file-system problems**, placed **last** because it is the parent class. If it were first, it would swallow the specific ones above.
11. `print(f"[warning] operating system problem: {err}")` - include Python's own message for unexpected cases.
12. `return default` - reached only if an `except` block ran (a success would already have returned). This gives the caller a safe fallback.
13. *(two blank lines)*
14. `print(repr(safe_read_text(WORKDIR / "number.txt")))` - test with an existing file; `repr` shows the exact string.
15. `print(repr(safe_read_text(WORKDIR / "ghost.txt", default="N/A")))` - test with a missing file and a custom default.

### Interpreting the output

```
'42'
[warning] ghost.txt was not found
'N/A'
```

The first call succeeded and returned the file content `'42'`. The second call printed a warning (from inside the function), then returned the default `'N/A'`. The calling code never crashes and never needs its own `try`/`except`.

## 4.8 Custom exceptions and `raise`

You can define your own exception classes to describe problems specific to your data, then `raise` them when validation fails. This produces clearer code than returning special values such as `-1` or `None`.

In [31]:
class InvalidRecordError(Exception):
    """Raised when a data row fails validation."""


def validate_row(row):
    if not row["name"].strip():
        raise InvalidRecordError("name is empty")
    try:
        age = int(row["age"])
    except ValueError:
        raise InvalidRecordError(f"age '{row['age']}' is not a number") from None
    if age < 0:
        raise InvalidRecordError("age is negative")
    return {"name": row["name"].strip(), "age": age}


tests = [
    {"name": "Amina", "age": "21"},
    {"name": "Tunde", "age": "abc"},
    {"name": " ", "age": "30"},
    {"name": "Chidi", "age": "-5"},
]

for row in tests:
    try:
        print("OK      :", validate_row(row))
    except InvalidRecordError as err:
        print("REJECTED:", err)

OK      : {'name': 'Amina', 'age': 21}
REJECTED: age 'abc' is not a number
REJECTED: name is empty
REJECTED: age is negative


### Line-by-line breakdown

1. `class InvalidRecordError(Exception):` - creates a new exception type by inheriting from `Exception`.
2. `"""Raised when a data row fails validation."""` - a docstring; it is enough to make a valid class body, so no other code is needed.
3. *(two blank lines)*
4. `def validate_row(row):` - a function that checks one CSV row (a dictionary) and returns a cleaned version.
5. `if not row["name"].strip():` - `.strip()` removes spaces; an empty string is falsy, so `not` is `True` when the name is blank.
6. `raise InvalidRecordError("name is empty")` - **raises** our custom error with a message, stopping the function immediately.
7. `try:` - protects the number conversion.
8. `age = int(row["age"])` - fails with `ValueError` for text such as `"abc"`.
9. `except ValueError:` - catches that conversion failure.
10. `raise InvalidRecordError(f"age '{row['age']}' is not a number") from None` - replaces the technical `ValueError` with our clearer error. `from None` hides the original error so the message stays clean.
11. `if age < 0:` - a logical check on the converted value.
12. `raise InvalidRecordError("age is negative")` - reject negative ages.
13. `return {"name": row["name"].strip(), "age": age}` - reached only if every check passed; returns cleaned data with `age` as a real integer.
14. *(two blank lines)*
15. `tests = [` ... `]` (spanning several lines) - four sample rows: one valid and three invalid in different ways.
16. *(blank line)*
17. `for row in tests:` - test each row.
18. `try:` - protected block around validation.
19. `print("OK      :", validate_row(row))` - runs only if no exception is raised.
20. `except InvalidRecordError as err:` - catches **only our custom error**, not unrelated bugs, which is exactly what we want.
21. `print("REJECTED:", err)` - shows the reason for rejection.

### Interpreting the output

```
OK      : {'name': 'Amina', 'age': 21}
REJECTED: age 'abc' is not a number
REJECTED: name is empty
REJECTED: age is negative
```

The valid row came back cleaned (age converted to `21`). Each invalid row produced a clear, human-readable reason. Because the loop catches `InvalidRecordError` specifically, a genuine programming bug (such as a missing key) would still surface instead of being silently hidden.

---
# Part 5: Mini Project, a CSV to JSON Pipeline with Full Error Handling

We combine everything: **read** a CSV, **validate** every row, collect the **good** and **bad** records separately, and **write** a JSON report. A single bad row must never stop the whole job.

In [32]:
raw_path = WORKDIR / "raw_users.csv"
raw_path.write_text(
    "name,age\n"
    "Amina,21\n"
    "Tunde,abc\n"
    ",30\n"
    "Chidi,-5\n"
    "Ngozi,28\n",
    encoding="utf-8",
)


def csv_to_json(csv_path, out_path):
    good, bad = [], []
    try:
        with open(csv_path, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for line_no, row in enumerate(reader, start=2):
                try:
                    good.append(validate_row(row))
                except InvalidRecordError as err:
                    bad.append({"line": line_no, "reason": str(err)})
    except FileNotFoundError:
        print("Input file not found:", Path(csv_path).name)
        return None

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump({"valid": good, "rejected": bad}, f, indent=2)
    return len(good), len(bad)


clean_path = WORKDIR / "clean_users.json"
summary = csv_to_json(raw_path, clean_path)
print("Valid, rejected:", summary)
print(clean_path.read_text(encoding="utf-8"))

print("Missing input ->", csv_to_json(WORKDIR / "nope.csv", WORKDIR / "x.json"))

Valid, rejected: (2, 3)
{
  "valid": [
    {
      "name": "Amina",
      "age": 21
    },
    {
      "name": "Ngozi",
      "age": 28
    }
  ],
  "rejected": [
    {
      "line": 3,
      "reason": "age 'abc' is not a number"
    },
    {
      "line": 4,
      "reason": "name is empty"
    },
    {
      "line": 5,
      "reason": "age is negative"
    }
  ]
}
Input file not found: nope.csv
Missing input -> None


### Line-by-line breakdown

1. `raw_path = WORKDIR / "raw_users.csv"` - path of the messy input file.
2. `raw_path.write_text(` - starts writing the test data. Adjacent string literals inside parentheses are **joined automatically** into one string, so we can lay out the file one row per line.
3. `"name,age\n"` - the header row.
4. `"Amina,21\n"` - valid row (file line 2).
5. `"Tunde,abc\n"` - invalid: age is not a number (line 3).
6. `",30\n"` - invalid: the name is empty (line 4).
7. `"Chidi,-5\n"` - invalid: negative age (line 5).
8. `"Ngozi,28\n",` - valid row (line 6).
9. `encoding="utf-8",` and `)` - finish the call.
10. *(two blank lines)*
11. `def csv_to_json(csv_path, out_path):` - the pipeline function.
12. `good, bad = [], []` - two empty lists created in one line: one for accepted records and one for rejected records.
13. `try:` - **outer** protection for file-level problems.
14. `with open(csv_path, "r", newline="", encoding="utf-8") as f:` - open the CSV correctly.
15. `reader = csv.DictReader(f)` - read rows as dictionaries.
16. `for line_no, row in enumerate(reader, start=2):` - the header is line 1, so the first data row is line 2. The counter lets us tell the user **exactly which line** is bad.
17. `try:` - **inner** protection for row-level problems.
18. `good.append(validate_row(row))` - validate and store the cleaned row if it passes.
19. `except InvalidRecordError as err:` - a bad row is caught here and the loop **continues** with the next row.
20. `bad.append({"line": line_no, "reason": str(err)})` - record the line number and the reason as text.
21. `except FileNotFoundError:` - outer handler for a missing input file.
22. `print("Input file not found:", Path(csv_path).name)` - report it.
23. `return None` - stop and signal failure.
24. *(blank line)*
25. `with open(out_path, "w", encoding="utf-8") as f:` - runs only if reading completed. Opens the JSON output file.
26. `json.dump({"valid": good, "rejected": bad}, f, indent=2)` - writes one JSON object holding both lists.
27. `return len(good), len(bad)` - returns a tuple of counts so callers can check the result.
28. *(two blank lines)*
29. `clean_path = WORKDIR / "clean_users.json"` - output path.
30. `summary = csv_to_json(raw_path, clean_path)` - run the pipeline.
31. `print("Valid, rejected:", summary)` - show the counts.
32. `print(clean_path.read_text(encoding="utf-8"))` - show the JSON report.
33. *(blank line)*
34. `print("Missing input ->", csv_to_json(WORKDIR / "nope.csv", WORKDIR / "x.json"))` - test the failure path with a file that does not exist.

### Interpreting the output

`Valid, rejected: (2, 3)` means two rows passed and three were rejected. The JSON report lists the valid people (Amina, 21 and Ngozi, 28) and three rejected entries with their line numbers and reasons: line 3 (`age 'abc' is not a number`), line 4 (`name is empty`) and line 5 (`age is negative`). The last test prints an `Input file not found: nope.csv` message followed by `Missing input -> None`.

The design lesson is **two layers of protection**: an inner `try`/`except` handles bad *rows* and keeps going, while an outer `try`/`except` handles bad *files* and stops cleanly.

---
# Part 6: Best Practices, Cheat Sheet and Exercises

## 6.1 A safer way to overwrite a file (write to a temporary file first)

Earlier we warned that opening a file with `"w"` erases it immediately. If your program crashes halfway through writing, you may lose the old data **and** the new data. The professional pattern is: write to a temporary file, then swap it into place in one step.

In [33]:
def save_json_safely(data, path):
    path = Path(path)
    temp_path = path.with_name(path.name + ".tmp")
    try:
        with open(temp_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        temp_path.replace(path)
    except (OSError, TypeError) as err:
        print("Save failed:", err)
        if temp_path.exists():
            temp_path.unlink()
        return False
    return True


target = WORKDIR / "settings.json"
print("First save  :", save_json_safely({"theme": "dark"}, target))
print("Second save :", save_json_safely({"theme": "light", "font": 14}, target))
print("Bad data    :", save_json_safely({"when": date(2026, 9, 28)}, target))
print("File content:", json.loads(target.read_text(encoding="utf-8")))
print("Leftover .tmp files:", list(WORKDIR.glob("*.tmp")))

First save  : True
Second save : True
Save failed: Object of type date is not JSON serializable
Bad data    : False
File content: {'theme': 'light', 'font': 14}
Leftover .tmp files: []


### Line-by-line breakdown

1. `def save_json_safely(data, path):` - takes the data to save and the final destination.
2. `path = Path(path)` - accepts strings or `Path` objects by converting to `Path`.
3. `temp_path = path.with_name(path.name + ".tmp")` - builds a sibling path such as `settings.json.tmp` in the same folder (the same folder matters, because replacing works reliably within one drive).
4. `try:` - protect the operation.
5. `with open(temp_path, "w", encoding="utf-8") as f:` - write to the **temporary** file, leaving the real file untouched.
6. `json.dump(data, f, indent=2)` - serialize into the temporary file. If this fails, the real file is still safe.
7. `temp_path.replace(path)` - swap the finished temporary file over the real one in a single quick operation. The real file is never half-written.
8. `except (OSError, TypeError) as err:` - catches file-system problems and unserializable data.
9. `print("Save failed:", err)` - report.
10. `if temp_path.exists():` - check whether a half-written temporary file was left behind.
11. `temp_path.unlink()` - `unlink()` deletes a file; we clean up after ourselves.
12. `return False` - tell the caller the save failed.
13. `return True` - reached only if nothing failed.
14. *(two blank lines)*
15. `target = WORKDIR / "settings.json"` - the destination file.
16. `print("First save  :", ...)` - saves a small settings dictionary.
17. `print("Second save :", ...)` - overwrites it with updated settings.
18. `print("Bad data    :", ...)` - tries to save a `date` object, which JSON cannot serialize, to trigger the failure path.
19. `print("File content:", ...)` - reads the settings file back to see what survived.
20. `print("Leftover .tmp files:", list(WORKDIR.glob("*.tmp")))` - confirms no temporary junk remains.

### Interpreting the output

The first two saves print `True`. The third prints `Save failed: Object of type date is not JSON serializable` and then `False`. The file content is still `{'theme': 'light', 'font': 14}`: **the failed save did not damage the good data from the second save**. That is the entire point of this pattern. The list of leftover temporary files is empty (`[]`).

## 6.2 Best practices checklist

1. **Always use `with`** so files are closed even when errors occur.
2. **Always set `encoding="utf-8"`** for text files, and `newline=""` for CSV files.
3. **Pick the mode deliberately**: `"r"` to read, `"a"` to add, `"w"` only when replacing is intended, `"x"` to protect existing files.
4. **Iterate over the file object** (`for line in f`) for large files instead of `read()`.
5. **Use `pathlib`** instead of manually gluing path strings together.
6. **Catch specific exceptions** (`FileNotFoundError`, `json.JSONDecodeError`) before general ones (`OSError`), and never use a bare `except:`.
7. **Keep the `try` block small**: only the risky lines belong there.
8. **Validate data after reading it**: CSV gives you strings, JSON can hold unexpected shapes.
9. **Use JSON for nested data**, **CSV for flat tables**, **JSON Lines for logs and streams**.
10. **Write to a temporary file first** when replacing important data.

## 6.3 Cheat sheet

| Task | Code |
|------|------|
| Read entire text file | `Path("a.txt").read_text(encoding="utf-8")` |
| Write text file | `Path("a.txt").write_text("hi", encoding="utf-8")` |
| Append text | `with open("a.txt", "a", encoding="utf-8") as f: f.write("x\n")` |
| Loop over lines | `for line in f:` |
| Read CSV as lists | `csv.reader(f)` |
| Read CSV as dicts | `csv.DictReader(f)` |
| Write CSV | `csv.writer(f).writerows(rows)` |
| JSON file to Python | `json.load(f)` |
| Python to JSON file | `json.dump(obj, f, indent=4)` |
| JSON string to Python | `json.loads(text)` |
| Python to JSON string | `json.dumps(obj)` |
| Check a path exists | `Path("a.txt").exists()` |
| Handle a missing file | `except FileNotFoundError:` |
| Handle bad JSON | `except json.JSONDecodeError:` |

## 6.4 Practice exercises

1. **Word counter:** write a function that reads a text file and returns a dictionary of how many times each word appears. Handle a missing file gracefully.
2. **Log filter:** given a log file where each line begins with `INFO`, `WARNING` or `ERROR`, copy only the `ERROR` lines into a new file, reading line by line.
3. **CSV summary:** read `students.csv`, add a new column called `grade` (A for 85 or above, B for 70 to 84, C otherwise) and write the result to `graded.csv` using `DictWriter`.
4. **Config manager:** create `load_config(path)` that returns settings from a JSON file, returns built-in defaults if the file is missing, and reports a clear message if the JSON is invalid.
5. **CSV and JSON converter:** write a function that converts any CSV file into a list of dictionaries saved as JSON, and another that converts it back.
6. **Challenge:** extend the Part 5 pipeline so rejected rows are also saved into a separate `rejected.csv` file with a `reason` column.

In [ ]:
import shutil

shutil.rmtree(WORKDIR)
print("Sandbox folder removed:", not WORKDIR.exists())

Sandbox folder removed: True


: 

### Line-by-line breakdown

1. `import shutil` - `shutil` is the standard library's "shell utilities" module for high-level file operations.
2. *(blank line)*
3. `shutil.rmtree(WORKDIR)` - **recursively deletes** the folder and everything inside it. Use with great care: it does not use a recycle bin.
4. `print("Sandbox folder removed:", not WORKDIR.exists())` - `exists()` is now `False`, and `not False` is `True`.

### Interpreting the output

`Sandbox folder removed: True` confirms the cleanup. To experiment again, re-run the notebook from the top and the sandbox will be rebuilt.

---
## Summary

- Files are opened with `open()` (or `Path` shortcuts) and should almost always be managed with **`with`**.
- **Modes** decide what happens to existing data: `"r"` reads, `"w"` replaces, `"a"` appends, `"x"` refuses to overwrite, and `"b"` handles raw bytes.
- The **`csv`** module handles quoting and delimiters correctly; everything read from CSV is text, so convert types yourself.
- The **`json`** module preserves nested structure; remember `dump`/`load` are for files and `dumps`/`loads` are for strings.
- **`try`/`except`/`else`/`finally`** turn crashes into controlled responses. Catch **specific** exceptions first, keep `try` blocks small and validate your data.

---

**Kajola Gbenga**  
Data Scientist & AI Engineer